# Lethe — Quickstart

**Link Inference Attacks for Evaluation of Edge Unlearning Methods**

This notebook does two things:

1. Runs a real Lethe benchmark configuration end-to-end (DBLP × GCN × high-centrality
   forget set) and reproduces the shape of Table 2 from the paper.
2. Shows you how to drop **your own unlearning method** into that same benchmark
   and have it scored against the published baselines.

It assumes the `lethe` environment is already active (see the README) and that this
notebook is being run from inside a clone of the repository.

> The demo config evaluates 3 unlearners (Original, Gold Model, Fine-tuning) instead of
> the 16 in the full benchmark.

## 0. Locate the repository

Configs resolve paths such as `resources/data` and `configs/snippets/` relative to the
repository root, so the working directory is moved there for the rest of the notebook.

In [1]:
import os
from pathlib import Path

root = Path.cwd()
while not (root / "main.py").exists() and root != root.parent:
    root = root.parent

assert (root / "lethe").is_dir(), "Run this notebook from inside the Lethe repository."
os.chdir(root)

import torch
print("Repository root:", root)
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

Repository root: /path/to/Lethe
torch: 2.13.0 | CUDA: False


## 1. Run the benchmark

`main.py` takes a single JSONC config that fully specifies the experiment: dataset,
forget-set construction, model, unlearners, and metrics.

The first run downloads DBLP (~15 MB) and trains the Original and Gold models. Both are
cached under `resources/cached/` by the config's `alias`, so every later run reuses them.

`SaveValues` *appends* to its output file, so `run_lethe` deletes any previous results
first — otherwise repeated runs accumulate duplicate records.

In [2]:
import subprocess, sys

CONFIG  = "configs/demo/DBLP_GCN_demo.jsonc"
RESULTS = "output/runs/demo/DBLP_GCN_demo.json"

def run_lethe(config, results_path):
    if os.path.exists(results_path):
        os.remove(results_path)

    proc = subprocess.Popen([sys.executable, "main.py", config],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Run failed with exit code {proc.returncode}")
    return results_path

run_lethe(CONFIG, RESULTS)
print("\nResults written to:", RESULTS)

2026-09-29 14:08:53 | INFO | 35090 - Creating Global Context for: configs/demo/DBLP_GCN_demo.jsonc
2026-09-29 14:08:53 | INFO | 35090 - Setting seeds to: 0
2026-09-29 14:08:53 | INFO | 35090 - REMOVAL TYPE SET AS edge
2026-09-29 14:08:53 | INFO | 35090 - Caching System: True.
2026-09-29 14:08:53 | INFO | 35090 - Instantiating: torch_geometric.datasets.CitationFull
{'root': 'resources/data', 'name': 'DBLP'}
2026-09-29 14:08:53 | INFO | 35090 - Created Configurable: lethe.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource
2026-09-29 14:08:53 | INFO | 35090 - {'class': 'lethe.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource', 'parameters': {'datasource': {'class': 'torch_geometric.datasets.CitationFull', 'parameters': {'root': 'resources/data', 'name': 'DBLP'}, 'preprocess': []}}}
2026-09-29 14:08:53 | INFO | 35090 - Instantiating: lethe.data.datasets.DataSplitterGraph.DataSplitterPercentage
2026-09-29 14:08:53 | INFO | 35090 - ['all', 'all_shuffled', '

2026-09-29 14:08:53 | INFO | 35090 - ['all', 'all_shuffled', '-', 'train_0', 'test', 'validation', 'train', 'forget', 'retain']
2026-09-29 14:08:53 | INFO | 35090 - ['all', 'all_shuffled', '-', 'train_0', 'test', 'validation', 'train', 'forget', 'retain']
2026-09-29 14:08:53 | INFO | 35090 - Created Configurable: lethe.data.datasets.DatasetManager.DatasetManager


2026-09-29 14:08:56 | INFO | 35090 - Loaded Instance from: resources/cached/DBLP_GCN_demo
2026-09-29 14:08:56 | INFO | 35090 - Created Configurable: lethe.model.TorchGraphModel.TorchGraphModel
2026-09-29 14:08:56 | INFO | 35090 - Global Predictor: <lethe.model.TorchGraphModel.TorchGraphModel object at 0x10c93df70>


2026-09-29 14:08:56 | INFO | 35090 - Created Configurable: lethe.unlearners.composite.Identity


2026-09-29 14:08:57 | INFO | 35090 - Created Configurable: lethe.unlearners.GoldModel.GoldModelGraph


...

2026-09-29 14:09:18 | INFO | 35090 - sklearn.metrics.accuracy_score on partition: "test", target model: unlearned, unlearned graph: True: 0.8253386004514672 of <lethe.model.TorchGraphModel.TorchGraphModel object at 0x17a2d1460>
2026-09-29 14:09:18 | INFO | 35090 - sklearn.metrics.accuracy_score on partition: "forget", target model: unlearned, unlearned graph: True: 0.8128580521960534 of <lethe.model.TorchGraphModel.TorchGraphModel object at 0x17a2d1460>
2026-09-29 14:09:18 | INFO | 35090 - sklearn.metrics.accuracy_score on partition: "test", target model: original, unlearned graph: False: 0.8244920993227991 of <lethe.model.TorchGraphModel.TorchGraphModel object at 0x12f7483e0>
[Edge Sampling] #Exist: 2643 | #Non-Exist: 2643
[Sample Exist] [(35, 38), (35, 462), (35, 39), (35, 3034), (38, 39)]
[Sample Non-Exist] [(6805, 15687), (5830, 8065), (5220, 12561), (13618, 17432), (15091, 17401)]
2026-09-29 14:09:18 | INFO | 35090 - LinkTeller: capped to 500 edges for evaluation


[Norm Stats] Exist Edges: mean=0.0049, std=0.0170, min=0.0000, max=0.1980
[Norm Stats] Non-Exist Edges: mean=0.0005, std=0.0008, min=0.0000, max=0.0060
[ROC Thresholds] Sample thresholds: [       inf 0.19796729 0.013328   0.01244581 0.00602715]
[ROC] FPR: [0. 0. 0. 0. 0.], TPR: [0.    0.002 0.1   0.104 0.122]
auc = 0.41197799999999996
2026-09-29 14:09:24 | INFO | 35090 - LinkTeller unlearn auc with sampler balanced: 0.41197799999999996


Attack 0 target posterior similarity
2026-09-29 14:09:25 | INFO | 35090 - Link Stealing Attack unlearned 0 exist/non_exist: 0.8709738428955316


Attack 0 target posterior similarity
2026-09-29 14:09:25 | INFO | 35090 - Link Stealing Attack 0 unlearned forget/non_exist: 0.7335295584212659



Results written to: output/runs/demo/DBLP_GCN_demo.json


## 2. Read the results

Each unlearner contributes one record. The file holds *concatenated* JSON objects
(`{...},{...},`) rather than a single array, so it is wrapped into a list before parsing.

Metric keys are verbose (`sklearn.metrics.accuracy_score.test.unlearned.on_graph:True`)
and the attack keys vary with the config's `target`, so columns are matched by substring
rather than by exact name.

In [3]:
import json
import pandas as pd

def load_results(path):
    raw = open(path).read().strip().rstrip(",")
    return pd.DataFrame(json.loads("[" + raw + "]"))

def summarise(df):
    def col(*needles, default=None):
        for c in df.columns:
            if all(n.lower() in c.lower() for n in needles):
                return df[c]
        return default

    out = pd.DataFrame({
        "Method":      df["unlearner"],
        "RunTime":     col("RunTime"),
        "Acc":         col("accuracy_score", "test", "unlearned"),
        "Acc(forget)": col("accuracy_score", "forget", "unlearned"),
        "LT AUC":      col("LinkTeller", "unlearn"),
        "LS AUC":      col("Link Stealing", "forget/non_exist"),
    })
    return out.round(4)

summarise(load_results(RESULTS))

,Method,RunTime,Acc,Acc(forget),LT AUC,LS AUC
0,Identity,0.3403,0.8251,0.8113,0.4154,0.7328
1,GoldModelGraph,3.2211,0.8259,0.8150,0.3865,0.7344
2,Finetuning,0.3602,0.8253,0.8129,0.4120,0.7335


**How to read this.** `Acc` barely moves across methods. That is the paper's central
negative result: accuracy cannot distinguish unlearning methods. The attack columns are
the informative ones, and **0.5 is the ideal value** (the adversary is guessing). An AUC
well above 0.5 means the forgotten edges are still recoverable from the model.

The Gold Model row is the reference: it is what genuine retraining achieves.

## 3. Add your own method

An unlearner subclasses `GraphUnlearner` and implements `__unlearn__`. The base class
hands you:

| Attribute | What it is |
|---|---|
| `self.predictor` | the trained model to modify (`.model`, `.optimizer`) |
| `self.dataset.partitions[...]` | the dataset splits (see the note below) |
| `self.task_loss(node_subset=...)` | the predictor's training loss on a subset |
| `self.infected_nodes(edges, hops)` | nodes a GNN of this depth can see the removed edges through |
| `self.hops` | receptive-field depth of the predictor |

Return the modified predictor. Nothing needs to be registered: the factory imports your
class by dotted path (`lethe/core/factory_base.py:39`), so a plain module at the
repository root is enough.

The example below runs gradient *ascent* on the nodes affected by the forget set, then a
short fine-tune on the retain set to recover utility. It exists to show the API surface: replace the body of `__unlearn__` with your own logic.

In [4]:
%%writefile my_method.py
from lethe.unlearners.graph_unlearners.GraphUnlearner import GraphUnlearner
from lethe.core.factory_base import get_instance_kvargs


class MyUnlearner(GraphUnlearner):

    def init(self):
        super().init()
        self.ascent_epochs = self.local.config['parameters']['ascent_epochs']
        self.repair_epochs = self.local.config['parameters']['repair_epochs']
        self.predictor.optimizer = get_instance_kvargs(
            self.local_config['parameters']['optimizer']['class'],
            {'params': self.predictor.model.parameters(),
             **self.local_config['parameters']['optimizer']['parameters']})

    def __unlearn__(self):
        forget_edges = self.dataset.partitions[self.forget_part]
        affected = self.infected_nodes(forget_edges, self.hops)
        repair_nodes = self.dataset.partitions[self.train_part]

        self.predictor.model.train()

        for epoch in range(self.ascent_epochs):
            self.predictor.optimizer.zero_grad()
            loss = -self.task_loss(node_subset=affected)
            loss.backward()
            self.predictor.optimizer.step()
            self.info(f'MyUnlearner ascent epoch={epoch} loss={loss.item():.4f}')

        for epoch in range(self.repair_epochs):
            self.predictor.optimizer.zero_grad()
            loss = self.task_loss(node_subset=repair_nodes)
            loss.backward()
            self.predictor.optimizer.step()
            self.info(f'MyUnlearner repair epoch={epoch} loss={loss.item():.4f}')

        return self.predictor

    def check_configuration(self):
        super().check_configuration()
        p = self.local.config['parameters']
        p['ascent_epochs'] = p.get('ascent_epochs', 3)
        p['repair_epochs'] = p.get('repair_epochs', 5)
        p['optimizer'] = p.get('optimizer', {'class': 'torch.optim.Adam',
                                             'parameters': {'lr': 0.0001}})

Overwriting my_method.py


Now append it to the config's `unlearners` list and re-run. A new config file is written
rather than editing the demo one, so the original stays reproducible, and results go to a
separate path so the two runs can be compared.

In [5]:
from jsonc_parser.parser import JsoncParser

cfg = JsoncParser.parse_file(CONFIG)

cfg["unlearners"].append({
    "class": "my_method.MyUnlearner",
    "parameters": {
        "ascent_epochs": 3,
        "repair_epochs": 5,
        "optimizer": {"class": "torch.optim.Adam", "parameters": {"lr": 0.0001}}
    }
})

MY_RESULTS = "output/runs/demo/DBLP_GCN_mymethod.json"
for m in cfg["evaluator"]["parameters"]["measures"]:
    if m["class"].endswith("SaveValues"):
        m["parameters"]["path"] = MY_RESULTS

MY_CONFIG = "configs/demo/DBLP_GCN_mymethod.jsonc"
with open(MY_CONFIG, "w") as f:
    json.dump(cfg, f, indent=2)

print("Wrote", MY_CONFIG)
print("Unlearners:", [u.get("class", u) for u in cfg["unlearners"]])

Wrote configs/demo/DBLP_GCN_mymethod.jsonc
Unlearners: [{'compose_idt': 'configs/snippets/u_id.json'}, 'lethe.unlearners.GoldModel.GoldModelGraph', 'lethe.unlearners.graph_unlearners.Finetuning.Finetuning', 'my_method.MyUnlearner']


The Original and Gold models are cached under the same `alias`, so this run only pays for
the unlearning methods themselves.

In [6]:
run_lethe(MY_CONFIG, MY_RESULTS)

summarise(load_results(MY_RESULTS))

2026-09-29 14:09:28 | INFO | 35139 - Creating Global Context for: configs/demo/DBLP_GCN_mymethod.jsonc
2026-09-29 14:09:28 | INFO | 35139 - Setting seeds to: 0
2026-09-29 14:09:28 | INFO | 35139 - REMOVAL TYPE SET AS edge
2026-09-29 14:09:28 | INFO | 35139 - Caching System: True.
2026-09-29 14:09:28 | INFO | 35139 - Instantiating: torch_geometric.datasets.CitationFull
{'root': 'resources/data', 'name': 'DBLP'}
2026-09-29 14:09:28 | INFO | 35139 - Created Configurable: lethe.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource
2026-09-29 14:09:28 | INFO | 35139 - {'class': 'lethe.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource', 'parameters': {'datasource': {'class': 'torch_geometric.datasets.CitationFull', 'parameters': {'root': 'resources/data', 'name': 'DBLP'}, 'preprocess': []}}}
2026-09-29 14:09:28 | INFO | 35139 - Instantiating: lethe.data.datasets.DataSplitterGraph.DataSplitterPercentage
2026-09-29 14:09:28 | INFO | 35139 - ['all', 'all_shuffled

2026-09-29 14:09:28 | INFO | 35139 - ['all', 'all_shuffled', '-', 'train_0', 'test', 'validation', 'train', 'forget', 'retain']
2026-09-29 14:09:28 | INFO | 35139 - ['all', 'all_shuffled', '-', 'train_0', 'test', 'validation', 'train', 'forget', 'retain']
2026-09-29 14:09:28 | INFO | 35139 - Created Configurable: lethe.data.datasets.DatasetManager.DatasetManager


2026-09-29 14:09:31 | INFO | 35139 - Loaded Instance from: resources/cached/DBLP_GCN_demo
2026-09-29 14:09:31 | INFO | 35139 - Created Configurable: lethe.model.TorchGraphModel.TorchGraphModel
2026-09-29 14:09:31 | INFO | 35139 - Global Predictor: <lethe.model.TorchGraphModel.TorchGraphModel object at 0x153604530>


2026-09-29 14:09:31 | INFO | 35139 - Created Configurable: lethe.unlearners.composite.Identity


2026-09-29 14:09:32 | INFO | 35139 - Created Configurable: lethe.unlearners.GoldModel.GoldModelGraph


...

2026-09-29 14:10:01 | INFO | 35139 - sklearn.metrics.accuracy_score on partition: "test", target model: unlearned, unlearned graph: True: 0.8259029345372461 of <lethe.model.TorchGraphModel.TorchGraphModel object at 0x32ddda5a0>
2026-09-29 14:10:01 | INFO | 35139 - sklearn.metrics.accuracy_score on partition: "forget", target model: unlearned, unlearned graph: True: 0.8101527689369828 of <lethe.model.TorchGraphModel.TorchGraphModel object at 0x32ddda5a0>
2026-09-29 14:10:01 | INFO | 35139 - sklearn.metrics.accuracy_score on partition: "test", target model: original, unlearned graph: False: 0.8244920993227991 of <lethe.model.TorchGraphModel.TorchGraphModel object at 0x31751c170>
[Edge Sampling] #Exist: 2643 | #Non-Exist: 2643
[Sample Exist] [(35, 38), (35, 462), (35, 39), (35, 3034), (38, 39)]
[Sample Non-Exist] [(6805, 15687), (5830, 8065), (5220, 12561), (13618, 17432), (15091, 17401)]
2026-09-29 14:10:01 | INFO | 35139 - LinkTeller: capped to 500 edges for evaluation


[Norm Stats] Exist Edges: mean=0.0047, std=0.0168, min=0.0000, max=0.1971
[Norm Stats] Non-Exist Edges: mean=0.0005, std=0.0008, min=0.0000, max=0.0060
[ROC Thresholds] Sample thresholds: [       inf 0.19711752 0.00756298 0.006664   0.00644723]
[ROC] FPR: [0. 0. 0. 0. 0.], TPR: [0.    0.002 0.12  0.124 0.126]
auc = 0.39020600000000005
2026-09-29 14:10:08 | INFO | 35139 - LinkTeller unlearn auc with sampler balanced: 0.39020600000000005


Attack 0 target posterior similarity
2026-09-29 14:10:09 | INFO | 35139 - Link Stealing Attack unlearned 0 exist/non_exist: 0.8696319452095934


Attack 0 target posterior similarity
2026-09-29 14:10:09 | INFO | 35139 - Link Stealing Attack 0 unlearned forget/non_exist: 0.7322058961355424


,Method,RunTime,Acc,Acc(forget),LT AUC,LS AUC
0,Identity,0.3425,0.8251,0.8113,0.4062,0.7328
1,GoldModelGraph,3.3311,0.8259,0.8150,0.4038,0.7344
2,Finetuning,0.3767,0.8253,0.8129,0.4041,0.7335
3,MyUnlearner,0.4453,0.8259,0.8102,0.3902,0.7322


## 4. Where to go next

See the README for the full configuration reference.